# Lecture 7 — Class Exercise
## Part-to-Whole: Hierarchical Visualization

> **Push to:** `week07/lecture07_exercise.ipynb`


In [3]:
import pandas as pd
import plotly.express as px
import numpy as np

# Dataset: Global Energy Mix by Country and Source
df = pd.read_csv(r"C:\Users\SHARIQ ALI\OneDrive\Desktop\UE\SEM 2\dataviz_push\week07\global_energy_mix.csv")

# Source type mapping — reuse from lecture
source_category = {
    'Coal': 'Fossil', 'Oil': 'Fossil', 'Natural Gas': 'Fossil',
    'Nuclear': 'Low-carbon', 'Hydro': 'Low-carbon',
    'Wind': 'Renewable', 'Solar': 'Renewable', 'Other Renewables': 'Renewable'
}
df['Source_Type'] = df['Source'].map(source_category)

print(f"Loaded: {len(df)} rows")
print(df.head(10))


Loaded: 103 rows
         Country         Region            Source  Share_pct     TWh  \
0  United States  North America              Coal         10  1015.0   
1  United States  North America               Oil         35  3220.0   
2  United States  North America       Natural Gas         34  3083.0   
3  United States  North America           Nuclear          9   798.0   
4  United States  North America             Hydro          3   339.0   
5  United States  North America              Wind          4   413.0   
6  United States  North America             Solar          3   325.0   
7  United States  North America  Other Renewables          2   229.0   
8          China           Asia              Coal         58  6929.1   
9          China           Asia               Oil         18  1620.0   

  Source_Type  
0      Fossil  
1      Fossil  
2      Fossil  
3  Low-carbon  
4  Low-carbon  
5   Renewable  
6   Renewable  
7   Renewable  
8      Fossil  
9      Fossil  


## Task 1 — Treemap: fossil fuel dependency by country

**What to build:** A treemap showing **fossil fuel TWh only**, broken down by Region → Country → Source (Coal / Oil / Natural Gas).

**Requirements:**
- Filter to fossil sources only before plotting
- Use `path=['Region', 'Country', 'Source']` for the hierarchy
- Colour encodes the fossil source type (Coal / Oil / Natural Gas) with a CVD-safe palette
- Show TWh values in labels — no percentages
- Grey out parent nodes (Region and Country level)
- Insight title naming which region or country is most fossil-dependent

> 💡 `df.loc[df['Source_Type'] == 'Fossil']`


In [4]:
# Task 1
# keep only fossil sources
df_fossil = df.loc[df['Source_Type'] == 'Fossil'].copy()

# find the most fossil-dependent region and country (by total fossil TWh)
region_totals = df_fossil.groupby('Region')['TWh'].sum().sort_values(ascending=False)
country_totals = df_fossil.groupby('Country')['TWh'].sum().sort_values(ascending=False)
top_region = region_totals.index[0]
top_country = country_totals.index[0]
print(region_totals)
print(country_totals.head())

# CVD-safe colours (Okabe-Ito) for the 3 fossil sources
# '(?)' is the colour plotly gives parent nodes -> make them grey
fossil_colors = {
    'Coal': '#D55E00',         # vermillion
    'Oil': '#0072B2',          # blue
    'Natural Gas': '#E69F00',  # orange
    '(?)': 'lightgrey'         # Region and Country boxes
}

fig1 = px.treemap(
    df_fossil,
    path=['Region', 'Country', 'Source'],
    values='TWh',
    color='Source',
    color_discrete_map=fossil_colors,
    title=f'{top_region} is the most fossil-dependent region ({region_totals.iloc[0]:,.0f} TWh); '
          f'{top_country} is the top country ({country_totals.iloc[0]:,.0f} TWh)'
)

# show label + TWh value, no percentages
fig1.update_traces(
    texttemplate='%{label}<br>%{value:,.0f} TWh',
    hovertemplate='%{label}<br>%{value:,.0f} TWh<extra></extra>'
)
fig1.update_layout(margin=dict(t=60, l=10, r=10, b=10))
fig1.show()

Region
Asia             35045.1
Europe           15387.0
Latin America    12376.0
North America    11131.0
Middle East       9671.0
Africa            8620.0
Oceania           8142.0
Name: TWh, dtype: float64
Country
Saudi Arabia    9671.0
Japan           9442.0
China           9421.1
South Africa    8620.0
South Korea     8252.0
Name: TWh, dtype: float64


## Task 2 — Treemap vs bar: low-carbon energy by country

**What to build:** Build **both** a treemap and a horizontal bar chart showing total low-carbon TWh (Nuclear + Hydro) per country. Then answer the question in a markdown cell below.

**Requirements:**
- Filter to `Source_Type == 'Low-carbon'` and aggregate TWh by country
- Treemap: single-level `path=['All', 'Country']` with a dummy root node labelled `'Low-carbon'`
- Bar chart: sorted by TWh, horizontal orientation, CVD-safe colour
- Both charts show TWh values, not percentages
- Insight title on the bar chart naming the leading country

In [5]:
# Task 2 — charts
# keep only low-carbon sources and add up TWh per country
df_low = df.loc[df['Source_Type'] == 'Low-carbon']
df_low = df_low.groupby('Country', as_index=False)['TWh'].sum()

# dummy root node for the treemap
df_low['All'] = 'Low-carbon'

# --- Treemap ---
fig2a = px.treemap(
    df_low,
    path=['All', 'Country'],
    values='TWh',
    color_discrete_sequence=['#0072B2'],  # CVD-safe blue
    title='Low-carbon energy (Nuclear + Hydro) by country'
)
fig2a.update_traces(
    texttemplate='%{label}<br>%{value:,.0f} TWh',
    hovertemplate='%{label}<br>%{value:,.0f} TWh<extra></extra>',
    root_color='lightgrey'
)
fig2a.update_layout(margin=dict(t=60, l=10, r=10, b=10))
fig2a.show()

# --- Horizontal bar chart ---
# sort ascending so the biggest bar ends up at the top
df_low_sorted = df_low.sort_values('TWh', ascending=True)
top_low = df_low_sorted.iloc[-1]

fig2b = px.bar(
    df_low_sorted,
    x='TWh',
    y='Country',
    orientation='h',
    text='TWh',
    color_discrete_sequence=['#0072B2'],  # same CVD-safe blue
    title=f"{top_low['Country']} leads low-carbon generation with {top_low['TWh']:,.0f} TWh"
)
fig2b.update_traces(texttemplate='%{text:,.0f} TWh', textposition='outside')
fig2b.update_layout(
    xaxis_title='Low-carbon generation (TWh)',
    yaxis_title='',
    plot_bgcolor='white'
)
fig2b.show()

### Task 2 — reflection

**Which chart tells the ranking story more clearly, and why?**

*YOUR ANSWER HERE*

## Task 3 — 100% stacked bar: which region relies most on coal?

**What to build:** A horizontal 100% stacked bar chart showing, for each **Region**, how its fossil energy splits between Coal, Oil and Natural Gas. This is the **same data as your Task 1 treemap**, visualised a second way.

**Requirements:**
- Reuse `df_fossil` from Task 1; aggregate TWh by `Region` × `Source` and convert to **shares of each region's fossil total**
- Horizontal bars, regions **sorted by coal share** (highest at the top)
- Coal is the segment on the baseline (first in `category_orders`) and the **only coloured segment** — keep it the same colour as in Task 1; Oil and Natural Gas in two greys
- Percentage labels on the coal segments only
- Insight title naming the most coal-dependent region and its coal share

> 💡 `df.groupby('Region')['TWh'].transform('sum')` gives each row its region total

In [6]:
# Task 3
# add up fossil TWh by Region and Source (reusing df_fossil from Task 1)
df_region = df_fossil.groupby(['Region', 'Source'], as_index=False)['TWh'].sum()

# share of each region's fossil total (in %)
df_region['Region_total'] = df_region.groupby('Region')['TWh'].transform('sum')
df_region['Share'] = df_region['TWh'] / df_region['Region_total'] * 100

# sort regions by coal share
coal = df_region.loc[df_region['Source'] == 'Coal'].sort_values('Share', ascending=False)
region_order = coal['Region'].tolist()   # highest coal share first
top_coal_region = coal.iloc[0]['Region']
top_coal_share = coal.iloc[0]['Share']

# label only the coal segments
df_region['Label'] = ''
df_region.loc[df_region['Source'] == 'Coal', 'Label'] = df_region['Share'].round(0).astype(int).astype(str) + '%'

# coal keeps its Task 1 colour, oil and gas are greys
stack_colors = {
    'Coal': '#D55E00',
    'Oil': '#A0A0A0',
    'Natural Gas': '#D3D3D3'
}

fig3 = px.bar(
    df_region,
    x='Share',
    y='Region',
    color='Source',
    orientation='h',
    text='Label',
    color_discrete_map=stack_colors,
    category_orders={'Source': ['Coal', 'Oil', 'Natural Gas'],
                     'Region': region_order},
    title=f'{top_coal_region} relies most on coal: {top_coal_share:.0f}% of its fossil energy'
)
fig3.update_traces(textposition='inside', insidetextanchor='middle')
fig3.update_layout(
    barmode='stack',
    xaxis=dict(title='Share of regional fossil energy (%)', range=[0, 100]),
    yaxis_title='',
    legend_title='',
    plot_bgcolor='white'
)
fig3.show()

### Task 3 — reflection

**Your Task 1 treemap and this stacked bar show the same fossil data. Name one question each chart answers that the other cannot.**

*YOUR ANSWER HERE*

The treemap answers questions about absolute size, for example “Which country burns the most coal in TWh?” or “How much of a region’s fossil energy comes from each country?”. The stacked bar can’t answer this because it only shows percentages and has no country level.

The 100% stacked bar answers questions about the mix, for example “Which region gets the largest share of its fossil energy from coal?”. The treemap can’t answer this well because regions have very different totals, so we’d have to compare coal areas inside rectangles of different sizes. The stacked bar puts every region on the same 0–100% scale, with coal on a shared baseline.